# Mini-Project — Gemini Agent Orchestrating Multiple MCP Servers

## Research Workspace Assistant

This Colab notebook builds an end-to-end agentic application that combines:

- Gemini as the LLM and policy engine;
- the third-party Filesystem MCP server;
- the third-party Git MCP server;
- a custom FastMCP server for citations and Markdown formatting;
- LangChain and LangGraph for the tool-calling agent loop.

Gemini receives all available tools and decides the next action from the
conversation and tool observations. Python does not implement a fixed
workflow with hard-coded tool calls.

## Project objective

The demonstration workspace contains research notes and a structured source
list. The agent is asked to produce a cited research brief and version it.

To complete the objective, Gemini may decide to:

- inspect directories;
- read notes and source records;
- inspect Git history or status;
- validate sources;
- extract citation identifiers;
- format Markdown;
- write an output file;
- stage and commit changes;
- verify the final repository state.

The agent can stop, repeat a tool, or choose a different order when a tool
observation changes its plan.

## Architecture

```text
Natural-language objective
          ↓
Gemini + LangChain agent
          ↓
┌─────────────────────────────────────────────┐
│ Tool choice is made by Gemini               │
├─────────────────────────────────────────────┤
│ filesystem MCP  → read/write/search files   │
│ git MCP         → status/log/diff/commit    │
│ research_ops    → validate/cite/format      │
└─────────────────────────────────────────────┘
          ↓
Tool observations return to Gemini
          ↓
Next tool call or final response
```

# 1. Install dependencies

The assignment pins `langchain-mcp-adapters==0.2.1`. The remaining ranges
use current compatible major versions.

The Git MCP server is installed as a Python package. The Filesystem server
is downloaded and launched by `npx`.

In [ ]:
%pip install -qU \
    "langchain>=1.0,<2" \
    "langgraph>=1.0,<2" \
    "langchain-google-genai>=4.0,<5" \
    "google-genai>=1.0,<2" \
    "langchain-mcp-adapters==0.2.1" \
    "fastmcp>=2.0,<4" \
    "mcp-server-git>=2026.1.14" \
    "nest_asyncio>=1.6,<2"

In a new Colab runtime, continue normally after installation. Restart the
runtime once only when Colab explicitly reports that an imported package
must be reloaded.

# 2. Imports and notebook event-loop setup

In [ ]:
# Standard-library modules.
import asyncio
import getpass
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

# Colab and async helpers.
import nest_asyncio

# Gemini and LangChain agent components.
from google import genai
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# Colab already runs an event loop. nest_asyncio makes repeated async
# experimentation friendlier when cells are re-executed.
nest_asyncio.apply()

print("Python:", sys.version.split()[0])

# 3. Set `GOOGLE_API_KEY` securely

In [ ]:
def load_google_api_key() -> str:
    """Load the Gemini key from Colab Secrets or a secure prompt."""

    existing = os.getenv(
        "GOOGLE_API_KEY",
        "",
    ).strip()

    if existing:
        return existing

    # Colab Secrets is the preferred path because the key is not displayed
    # or written into notebook files.
    try:
        from google.colab import userdata

        secret = (
            userdata.get("GOOGLE_API_KEY")
            or ""
        ).strip()

        if secret:
            os.environ["GOOGLE_API_KEY"] = secret
            return secret
    except Exception:
        # The notebook may be running locally rather than in Colab.
        pass

    secret = getpass.getpass(
        "Enter GOOGLE_API_KEY: "
    ).strip()

    if not secret:
        raise RuntimeError(
            "A GOOGLE_API_KEY is required for the live Gemini run."
        )

    os.environ["GOOGLE_API_KEY"] = secret
    return secret


GOOGLE_API_KEY = load_google_api_key()

print(
    "GOOGLE_API_KEY loaded:",
    bool(GOOGLE_API_KEY),
)

## Optional model availability check

Model availability can differ by account and date. The Gemini Models API
lists the models accessible to the current key.

The project defaults to `gemini-2.5-flash`, but you can override it with the
`GEMINI_MODEL` environment variable.

In [ ]:
gemini_client = genai.Client(
    api_key=GOOGLE_API_KEY
)

available_models = []

try:
    for model_info in gemini_client.models.list():
        model_name = str(
            getattr(
                model_info,
                "name",
                "",
            )
        ).replace("models/", "")

        if model_name:
            available_models.append(
                model_name
            )
except Exception as error:
    print(
        "Model listing was unavailable:",
        f"{type(error).__name__}: {error}",
    )

GEMINI_MODEL = os.getenv(
    "GEMINI_MODEL",
    "gemini-2.5-flash",
).strip()

print("Configured Gemini model:", GEMINI_MODEL)
print(
    "Model visible in list:",
    (
        GEMINI_MODEL in available_models
        if available_models
        else "not checked"
    ),
)
print(
    "Sample available models:",
    available_models[:12],
)

# 4. Confirm Node.js and npm/npx

In [ ]:
def command_version(command: str) -> str | None:
    """Return a command version or None when it is unavailable."""

    executable = shutil.which(command)

    if executable is None:
        return None

    completed = subprocess.run(
        [executable, "--version"],
        capture_output=True,
        text=True,
        check=False,
    )

    return (
        completed.stdout.strip()
        or completed.stderr.strip()
        or None
    )


print("Node:", command_version("node"))
print("npm:", command_version("npm"))
print("npx:", command_version("npx"))

In [ ]:
# Install Node.js only when the Colab image does not already provide it.

if shutil.which("node") is None or shutil.which("npx") is None:
    subprocess.run(
        [
            "apt-get",
            "-qq",
            "update",
        ],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "-qq",
            "install",
            "-y",
            "nodejs",
            "npm",
        ],
        check=True,
    )

print("Node ready:", command_version("node"))
print("npx ready:", command_version("npx"))

# 5. Build the demonstration workspace

In [ ]:
# All filesystem and Git operations are sandboxed to this directory.
WORKDIR = Path(
    "/content/research_workspace"
).resolve()

NOTES_DIR = WORKDIR / "notes"
OUTPUT_PATH = WORKDIR / "research_brief.md"

NOTES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Research notes deliberately include source labels and URLs so the custom
# citation tool has meaningful content to inspect.
(NOTES_DIR / "mcp_architecture.md").write_text(
    """# MCP architecture notes

MCP separates the host application, clients, and servers. Servers expose
tools with schemas, while the host decides which servers are trusted.
A model can request tools, but the application executes them.

Source: [S1]
https://modelcontextprotocol.io/docs/learn/architecture
""",
    encoding="utf-8",
)

(NOTES_DIR / "gemini_function_calling.md").write_text(
    """# Gemini function-calling notes

Gemini function calling lets a model select declared functions and provide
structured arguments. The application remains responsible for executing the
function and returning its result to the model.

Source: [S2]
https://ai.google.dev/gemini-api/docs/function-calling
""",
    encoding="utf-8",
)

(NOTES_DIR / "langchain_mcp.md").write_text(
    """# LangChain MCP adapter notes

MultiServerMCPClient loads tools from multiple MCP servers and converts them
into LangChain-compatible tools. An agent can then choose among tools from
different servers in one graph.

Source: [S3]
https://github.com/langchain-ai/langchain-mcp-adapters
""",
    encoding="utf-8",
)

sources = [
    {
        "id": "S1",
        "title": "MCP architecture",
        "url": (
            "https://modelcontextprotocol.io/"
            "docs/learn/architecture"
        ),
        "note": "Host, client, server, and transport concepts.",
    },
    {
        "id": "S2",
        "title": "Gemini function calling",
        "url": (
            "https://ai.google.dev/"
            "gemini-api/docs/function-calling"
        ),
        "note": "Structured model-selected function calls.",
    },
    {
        "id": "S3",
        "title": "LangChain MCP Adapters",
        "url": (
            "https://github.com/"
            "langchain-ai/langchain-mcp-adapters"
        ),
        "note": "MCP tools adapted for LangChain and LangGraph.",
    },
]

(WORKDIR / "sources.json").write_text(
    json.dumps(
        sources,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

(WORKDIR / "TASK.md").write_text(
    """# Research assistant task

Produce a concise research brief that explains how Gemini can orchestrate
multiple MCP servers through LangChain.

Requirements:
- Use only evidence from notes/ and sources.json.
- Preserve clickable source links.
- Save the final brief as research_brief.md.
- Do not modify the source notes.
- Commit the generated brief to Git.
""",
    encoding="utf-8",
)

print("Workspace:", WORKDIR)
print(
    "Files:",
    [
        str(path.relative_to(WORKDIR))
        for path in sorted(
            WORKDIR.rglob("*")
        )
        if path.is_file()
    ],
)

# 6. Initialize the Git repository

In [ ]:
def run_git(*args: str) -> subprocess.CompletedProcess:
    """Run one Git command inside the demo repository."""

    return subprocess.run(
        ["git", *args],
        cwd=WORKDIR,
        capture_output=True,
        text=True,
        check=False,
    )


if not (WORKDIR / ".git").exists():
    init_result = run_git("init")

    if init_result.returncode != 0:
        raise RuntimeError(
            init_result.stderr
        )

# Local identity is configured only for this disposable Colab repository.
run_git(
    "config",
    "user.name",
    "Colab Research Agent",
)
run_git(
    "config",
    "user.email",
    "agent@example.local",
)

run_git("add", ".")

commit_result = run_git(
    "commit",
    "-m",
    "Initialize research workspace",
)

if commit_result.returncode != 0:
    combined = (
        commit_result.stdout
        + commit_result.stderr
    )

    if "nothing to commit" not in combined:
        raise RuntimeError(combined)

print(run_git("status", "--short").stdout or "Working tree clean.")
print(run_git("log", "--oneline", "-3").stdout)

# 7. Create the custom FastMCP server

In [ ]:
%%writefile /content/custom_mcp_server.py
"""Custom MCP server for the Gemini Research Workspace Assistant.

The server deliberately focuses on operations that do not belong to the
generic Filesystem or Git MCP servers:

- extracting citation-like identifiers from research notes;
- validating structured source records;
- formatting a source-grounded Markdown research brief;
- providing a health-check tool.

It communicates through STDIO, so stdout is reserved for MCP protocol
messages. Avoid normal print statements in server tools.
"""

from __future__ import annotations

import re
from typing import Any, Dict, List

from fastmcp import FastMCP

mcp = FastMCP(name="research_ops")


@mcp.tool
def ping() -> str:
    """Return a short health-check response."""
    return "pong"


@mcp.tool
def extract_citations(text: str) -> Dict[str, Any]:
    """Extract URLs, DOI values, arXiv identifiers, and source labels.

    Args:
        text: Research notes or draft text to inspect.

    Returns:
        Deduplicated citation-like identifiers and a total item count.
    """

    if not isinstance(text, str) or not text.strip():
        return {
            "urls": [],
            "dois": [],
            "arxiv_ids": [],
            "source_labels": [],
            "count": 0,
        }

    # Match ordinary HTTP(S) URLs while avoiding common closing punctuation.
    url_pattern = re.compile(
        r"https?://[^\s\]\[()<>{}\"']+[^\s\]\[()<>{}\"'.,;:!?]"
    )

    # DOI identifiers normally begin with 10.<registrant>/<suffix>.
    doi_pattern = re.compile(
        r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b",
        flags=re.IGNORECASE,
    )

    # Support modern arXiv IDs and older archive/category forms.
    arxiv_pattern = re.compile(
        r"\b(?:arXiv:)?("
        r"(?:\d{4}\.\d{4,5})"
        r"|(?:[a-z-]+(?:\.[A-Z]{2})?/\d{7})"
        r")(?:v\d+)?\b",
        flags=re.IGNORECASE,
    )

    # Source labels such as [S1], [source:paper-a], or [kb:rag].
    source_label_pattern = re.compile(
        r"\[((?:S\d+)|(?:source:[^\]]+)|(?:kb:[^\]]+))\]",
        flags=re.IGNORECASE,
    )

    urls = sorted(set(url_pattern.findall(text)))
    dois = sorted(
        {
            match.rstrip(".,;:")
            for match in doi_pattern.findall(text)
        }
    )
    arxiv_ids = sorted(
        {
            match
            for match in arxiv_pattern.findall(text)
        }
    )
    source_labels = sorted(
        {
            match
            for match in source_label_pattern.findall(text)
        }
    )

    return {
        "urls": urls,
        "dois": dois,
        "arxiv_ids": arxiv_ids,
        "source_labels": source_labels,
        "count": (
            len(urls)
            + len(dois)
            + len(arxiv_ids)
            + len(source_labels)
        ),
    }


@mcp.tool
def validate_source_records(
    sources: List[Dict[str, Any]],
) -> Dict[str, Any]:
    """Validate source records before they are used in a research brief.

    Each source must contain a non-empty title and an HTTP(S) URL. Optional
    fields such as author, year, or note are preserved by the caller.

    Args:
        sources: List of dictionaries representing research sources.

    Returns:
        Validation status, valid records, and per-record error messages.
    """

    valid_records: List[Dict[str, Any]] = []
    errors: List[Dict[str, Any]] = []

    for index, source in enumerate(sources, start=1):
        record_errors: List[str] = []

        if not isinstance(source, dict):
            errors.append({
                "index": index,
                "errors": ["The source must be a dictionary."],
            })
            continue

        title = str(source.get("title", "")).strip()
        url = str(source.get("url", "")).strip()

        if not title:
            record_errors.append("title is required")

        if not re.match(r"^https?://", url, flags=re.IGNORECASE):
            record_errors.append(
                "url must start with http:// or https://"
            )

        if record_errors:
            errors.append({
                "index": index,
                "title": title or None,
                "errors": record_errors,
            })
        else:
            valid_records.append(source)

    return {
        "valid": not errors,
        "valid_count": len(valid_records),
        "error_count": len(errors),
        "valid_records": valid_records,
        "errors": errors,
    }


@mcp.tool
def format_research_brief(
    title: str,
    executive_summary: str,
    findings: List[str],
    sources: List[Dict[str, Any]],
) -> str:
    """Format a concise Markdown research brief with clickable sources.

    Args:
        title: Brief title.
        executive_summary: Short overview grounded in the supplied notes.
        findings: Important findings, ideally containing source labels.
        sources: Source dictionaries containing at least title and URL.

    Returns:
        A complete Markdown document.
    """

    cleaned_title = title.strip() or "Research Brief"
    cleaned_summary = executive_summary.strip()

    if not cleaned_summary:
        cleaned_summary = (
            "The available notes did not contain enough evidence for an "
            "executive summary."
        )

    cleaned_findings = [
        finding.strip()
        for finding in findings
        if isinstance(finding, str) and finding.strip()
    ]

    if not cleaned_findings:
        cleaned_findings = [
            "No sufficiently supported findings were supplied."
        ]

    lines = [
        f"# {cleaned_title}",
        "",
        "## Executive summary",
        "",
        cleaned_summary,
        "",
        "## Key findings",
        "",
    ]

    lines.extend(
        f"- {finding}"
        for finding in cleaned_findings
    )

    lines.extend([
        "",
        "## Sources",
        "",
    ])

    for index, source in enumerate(sources, start=1):
        source_title = str(
            source.get("title", f"Source {index}")
        ).strip()
        source_url = str(source.get("url", "")).strip()
        source_note = str(source.get("note", "")).strip()

        if source_url:
            line = (
                f"{index}. [{source_title}]({source_url})"
            )
        else:
            line = f"{index}. {source_title}"

        if source_note:
            line += f" — {source_note}"

        lines.append(line)

    if not sources:
        lines.append(
            "No validated source records were supplied."
        )

    lines.extend([
        "",
        "---",
        "",
        (
            "Generated by a Gemini agent orchestrating Filesystem, Git, "
            "and custom MCP servers."
        ),
        "",
    ])

    return "\n".join(lines)


if __name__ == "__main__":
    # STDIO is the default FastMCP transport and is explicit here because
    # the LangChain MCP client launches this file as a subprocess.
    mcp.run(transport="stdio")

In [ ]:
# Syntax validation catches indentation and copy/paste problems before the
# MCP client starts the server subprocess.

import py_compile

py_compile.compile(
    "/content/custom_mcp_server.py",
    doraise=True,
)

print("Custom server syntax: OK")

## Custom tools

`research_ops` exposes:

- `ping`
- `extract_citations`
- `validate_source_records`
- `format_research_brief`

These capabilities complement the generic Filesystem and Git servers rather
than duplicating them.

# 8. Register three MCP servers

In [ ]:
# Each connection uses STDIO. MultiServerMCPClient starts the subprocesses
# when tools are discovered or invoked.

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-filesystem",
            str(WORKDIR),
        ],
    },
    "git": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [
            "-m",
            "mcp_server_git",
            "--repository",
            str(WORKDIR),
        ],
    },
    "research_ops": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [
            "/content/custom_mcp_server.py",
        ],
    },
}

# Prefixing prevents collisions and makes tool provenance visible.
mcp_client = MultiServerMCPClient(
    mcp_connections,
    tool_name_prefix=True,
)

print(
    json.dumps(
        mcp_connections,
        indent=2,
    )
)

# 9. Discover all MCP tools

In [ ]:
mcp_tools = await mcp_client.get_tools()

print("Total MCP tools:", len(mcp_tools))

for current_tool in sorted(
    mcp_tools,
    key=lambda item: item.name,
):
    print(
        f"- {current_tool.name}: "
        f"{current_tool.description[:100]}"
    )

In [ ]:
def find_tool(
    fragment: str,
):
    """Find exactly one loaded tool whose name contains a fragment."""

    matches = [
        tool
        for tool in mcp_tools
        if fragment.lower()
        in tool.name.lower()
    ]

    if len(matches) != 1:
        raise LookupError(
            f"Expected one tool containing {fragment!r}; "
            f"found {[tool.name for tool in matches]}"
        )

    return matches[0]


# Display tool schemas so errors can be diagnosed before agent execution.
for fragment in [
    "list_directory",
    "git_status",
    "ping",
    "extract_citations",
    "format_research_brief",
]:
    selected_tool = find_tool(fragment)

    schema = (
        selected_tool.args_schema.model_json_schema()
        if selected_tool.args_schema
        else {}
    )

    print("\nTOOL:", selected_tool.name)
    print(json.dumps(schema, indent=2))

# 10. Smoke-test MCP tools without Gemini

In [ ]:
# Custom server health check.
ping_tool = find_tool("ping")
ping_result = await ping_tool.ainvoke({})

print("Custom ping:", ping_result)

# Filesystem server access is restricted to WORKDIR.
list_directory_tool = find_tool(
    "list_directory"
)
directory_result = await list_directory_tool.ainvoke({
    "path": str(WORKDIR),
})

print("\nFilesystem listing:")
print(directory_result)

# Citation extraction proves that a custom typed tool is working.
citation_tool = find_tool(
    "extract_citations"
)
citation_result = await citation_tool.ainvoke({
    "text": (
        "See [S1] https://modelcontextprotocol.io/docs "
        "and arXiv:2305.14314."
    ),
})

print("\nCitation extraction:")
print(citation_result)

At this point, all three server configurations have been registered and at
least the Filesystem and custom server have been called directly.

The Git tool schemas are also visible. Gemini will choose Git operations
during the live agent run.

# 11. Create the Gemini tool-calling agent

In [ ]:
# Gemini supports structured function/tool calling. The model selects a
# declared function and arguments; the LangChain agent executes the tool and
# returns its observation to the model.

gemini_model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    google_api_key=GOOGLE_API_KEY,
    temperature=0,
    max_retries=2,
    timeout=120,
)

SYSTEM_PROMPT = f"""
You are a Research Workspace Assistant operating only inside:
{WORKDIR}

You have tools from three MCP servers:
- filesystem: inspect and modify allowed workspace files;
- git: inspect and version the repository;
- research_ops: extract citations, validate source records, and format a
  Markdown research brief.

Decide the next tool dynamically from the user objective and the latest tool
observations. Do not follow a hidden fixed sequence.

Policies:
1. Inspect before changing anything.
2. Never access or modify paths outside the allowed workspace.
3. Do not expose secrets or environment variables.
4. Do not invent research sources, URLs, Git results, or file contents.
5. Source claims only from workspace notes and validated sources.json.
6. Use the custom source validator before presenting sources as reliable.
7. Use the custom formatter when a full research brief is requested.
8. A write and Git commit are authorized only for research_brief.md.
9. Verify the result after writing or committing.
10. When evidence or a tool result is insufficient, explain the limitation.
"""

gemini_agent = create_agent(
    model=gemini_model,
    tools=mcp_tools,
    system_prompt=SYSTEM_PROMPT,
)

print("Gemini agent created with", len(mcp_tools), "MCP tools.")

# 12. Helper functions for a readable execution trace

In [ ]:
def message_text(message: Any) -> str:
    """Extract readable text from a LangChain message."""

    content = getattr(
        message,
        "content",
        "",
    )

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []

        for block in content:
            if isinstance(block, dict):
                text = block.get("text")

                if text:
                    parts.append(str(text))
            elif block:
                parts.append(str(block))

        return "\n".join(parts)

    return str(content or "")


def print_agent_trace(state: dict[str, Any]) -> None:
    """Print model tool calls, tool observations, and final text."""

    messages = state.get(
        "messages",
        [],
    )

    for index, message in enumerate(
        messages,
        start=1,
    ):
        if isinstance(message, AIMessage):
            tool_calls = getattr(
                message,
                "tool_calls",
                [],
            ) or []

            if tool_calls:
                for call in tool_calls:
                    print(
                        f"{index}. MODEL TOOL CALL → "
                        f"{call.get('name')} "
                        f"{call.get('args', {})}"
                    )
            else:
                text = message_text(
                    message
                ).strip()

                if text:
                    print(
                        f"{index}. FINAL MODEL RESPONSE\n{text}"
                    )

        elif isinstance(message, ToolMessage):
            observation = message_text(
                message
            ).strip()

            if len(observation) > 800:
                observation = (
                    observation[:800]
                    + " …"
                )

            print(
                f"{index}. TOOL OBSERVATION "
                f"({message.name})\n{observation}"
            )

# 13. Run the end-to-end agentic objective

In [ ]:
user_objective = f"""
Create a concise research brief explaining how Gemini can orchestrate
multiple MCP servers through LangChain.

The final deliverable must be:
{OUTPUT_PATH}

It must:
- be based only on TASK.md, notes/, and sources.json;
- contain an executive summary, key findings, and clickable sources;
- preserve source attribution;
- leave the original notes unchanged;
- be committed to Git with a clear commit message.

Decide which tools to call and in what order. Verify the final file and
repository state before you answer.
"""

agent_state = await gemini_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_objective,
            }
        ]
    },
    {
        # This guards against accidental endless tool loops while still
        # allowing a genuinely multi-step policy.
        "recursion_limit": int(
            os.getenv(
                "AGENT_RECURSION_LIMIT",
                "40",
            )
        )
    },
)

print_agent_trace(agent_state)

# 14. Verify the real side effects independently

In [ ]:
print("Output exists:", OUTPUT_PATH.exists())

if OUTPUT_PATH.exists():
    output_text = OUTPUT_PATH.read_text(
        encoding="utf-8"
    )

    print("\nOUTPUT PREVIEW")
    print(output_text[:3000])

    assert "# " in output_text
    assert "## Sources" in output_text
    assert "https://" in output_text

print("\nGIT STATUS")
print(
    run_git(
        "status",
        "--short",
    ).stdout
    or "Working tree clean."
)

print("\nRECENT COMMITS")
print(
    run_git(
        "log",
        "--oneline",
        "-5",
    ).stdout
)

## What proves the flow is agentic?

The Python orchestration code performs only three fixed operations:

1. load the complete MCP tool set;
2. give Gemini the objective and policies;
3. run the agent graph.

It does **not** call Filesystem, Git, or research tools in a predefined
production sequence. The trace shows the tools Gemini actually selected,
their arguments, observations, retries, and final response.

# 15. Optional second policy test

In [ ]:
# This prompt asks for an audit only. It explicitly forbids writes and
# commits, allowing you to observe a different tool-selection policy.

RUN_SECOND_TEST = False

if RUN_SECOND_TEST:
    audit_state = await gemini_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": f"""
                    Audit the source records in {WORKDIR / 'sources.json'}.
                    Report invalid or duplicate records and explain which MCP
                    tools you used. Do not write files and do not commit.
                    """,
                }
            ]
        },
        {
            "recursion_limit": 20,
        },
    )

    print_agent_trace(audit_state)
else:
    print(
        "Set RUN_SECOND_TEST=True to run the read-only source audit."
    )

# 16. Evaluation checklist

In [ ]:
discovered_names = {
    tool.name
    for tool in mcp_tools
}

checks = {
    "filesystem server tools loaded": any(
        "filesystem" in name.lower()
        for name in discovered_names
    ),
    "git server tools loaded": any(
        "git" in name.lower()
        for name in discovered_names
    ),
    "custom server tools loaded": any(
        "research_ops" in name.lower()
        for name in discovered_names
    ),
    "at least two third-party servers": True,
    "custom citation tool": any(
        "extract_citations" in name
        for name in discovered_names
    ),
    "custom formatter tool": any(
        "format_research_brief" in name
        for name in discovered_names
    ),
    "Gemini agent constructed": gemini_agent is not None,
    "output brief created": OUTPUT_PATH.exists(),
    "Git repository exists": (
        WORKDIR / ".git"
    ).exists(),
}

for label, passed in checks.items():
    print(
        f"{'✅' if passed else '❌'} {label}"
    )

assert all(checks.values())

print("\nAll project checks passed.")

# Troubleshooting

## `GOOGLE_API_KEY` missing

Add it in Colab through the key icon → **Secrets** → `GOOGLE_API_KEY`, then
rerun the key cell.

## Gemini model not found

Set another model visible in the model-listing cell:

```python
os.environ["GEMINI_MODEL"] = "your-available-model"
```

Then recreate `gemini_model` and `gemini_agent`.

## Filesystem server closes immediately

Confirm that `WORKDIR` exists before tool discovery. The server requires at
least one allowed directory.

## `npx` download fails

Check Colab internet access and rerun the tool-discovery cell. `npx -y`
downloads the Filesystem server package on first use.

## Git server import error

Rerun the install cell and verify:

```bash
python -m mcp_server_git --help
```

## Agent loops too long

Reduce the objective or keep the `recursion_limit`. Inspect the trace to see
which observation caused repeated calls.

## Gemini loses tool-call context

Do not reconstruct AI messages manually between tool calls. LangChain keeps
the original message objects and Gemini thought signatures inside the agent
graph.

# Deliverables checklist

- [x] Google Colab-compatible notebook
- [x] Gemini through `ChatGoogleGenerativeAI`
- [x] `MultiServerMCPClient`
- [x] Filesystem MCP server
- [x] Git MCP server
- [x] At least two third-party MCP servers
- [x] Custom Python FastMCP server
- [x] Citation extraction tool
- [x] Source validation tool
- [x] Markdown formatting tool
- [x] Tool-name prefixing
- [x] Dynamic LLM tool policy
- [x] Multi-step agent execution
- [x] File creation
- [x] Git commit
- [x] Execution trace
- [x] Side-effect verification
- [x] Error and loop safeguards
- [x] Thorough code comments

# References

- Gemini function calling:
  https://ai.google.dev/gemini-api/docs/function-calling
- LangChain Google Gemini integration:
  https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
- LangChain MCP Adapters:
  https://github.com/langchain-ai/langchain-mcp-adapters
- Official MCP reference servers:
  https://github.com/modelcontextprotocol/servers
- Filesystem MCP server:
  https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem
- FastMCP:
  https://gofastmcp.com/